# Novos experimentos
Pipeline adaptado nesta entrega: audita PTS, detecta o drone, estima Z na ROI e integra telemetria quando informada. A execução completa com modelos ainda precisa ser validada em Colab. Não herda automaticamente a geometria nem a regressão do Farol.

## Abrir o pacote no Colab
O código abaixo clona a branch de preparação no ambiente do Colab. Ajuste ROOT para uma cópia local já existente. Para inferência, selecione GPU no ambiente de execução; a reprodução dos CSVs usa CPU.

In [ ]:
from pathlib import Path
import sys, subprocess, os, json
ROOT = Path('/content/Estimacao_Distancia_Drone_TCC') if Path('/content').exists() else Path.cwd()
if not (ROOT/'README.md').is_file():
    subprocess.run(['git','clone','--branch','preparacao-tcc-2026-10-17','https://github.com/icarodeferreira/Estimacao_Distancia_Drone_TCC.git',str(ROOT)],check=True)
assert (ROOT/'scripts/reproduzir_farol.py').is_file(), 'Ajuste ROOT para a raiz do repositório.'
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-inference.txt'],check=True)

In [ ]:
cfg=json.loads(Path('configs/novo.json').read_text())
cfg['video']=str(ROOT/'inputs/video/novo.mov')
cfg['experiment_id']='novo_auditoria_001'
Path('configs/execucao.json').write_text(json.dumps(cfg,indent=2))

In [ ]:
subprocess.run([sys.executable,'scripts/novo_experimento.py','configs/execucao.json','--audit-only'],check=True)

## Preencher aquisição e referências
Use a resolução exibida na auditoria. Preencha lente, zoom e identificação real da câmera. Os limiares de 10×5 pixels são valores iniciais do Farol; justifique a escolha para outro vídeo. Sem geometria e sincronização, as métricas de distância ficam indisponíveis.

In [ ]:
cfg['experiment_id']='novo_piloto_001'
cfg['field'].update(camera=None,lens=None,zoom=None,fixed_camera=None,constant_zoom=None) # preencher
cfg['measurement_resolution']=None # [largura, altura] após auditoria
# cfg['telemetry']='inputs/telemetry/voo.csv'
# cfg['telemetry_columns']={'t':'tempo_s','height':'altura_m','radius':'raio_m'}
# cfg['telemetry_units']={'height':'m','radius':'m'}
# cfg['sync']['events']=[{'video_s':..., 'telemetry_s':...}]
# Consulte docs/GUIA_NOVOS_EXPERIMENTOS.md para geometria nominal, angular ou ENU.
# cfg['geometry']['mode']='none' preserva ausência de referência.
Path('configs/execucao.json').write_text(json.dumps(cfg,indent=2))

## Instalar Depth Pro
O instalador registra a revisão efetiva e o SHA-256. Para repetir outra execução, passe `--revision COMMIT_REGISTRADO`. Não conhecemos o commit histórico exato.

In [ ]:
subprocess.run([sys.executable,'scripts/instalar_depth_pro.py'],check=True)

In [ ]:
subprocess.run([sys.executable,'scripts/novo_experimento.py','configs/execucao.json','--pilot-s','10'],check=True)

## Conferir piloto antes da execução completa
Inspecione as caixas e as saídas ausentes. Cada execução usa identificador próprio para preservar resultados.

In [ ]:
EXECUTAR_COMPLETO=False
if EXECUTAR_COMPLETO:
    cfg['experiment_id']='novo_completo_001'
    Path('configs/execucao.json').write_text(json.dumps(cfg,indent=2))
    subprocess.run([sys.executable,'scripts/novo_experimento.py','configs/execucao.json'],check=True)

In [ ]:
p=Path('outputs')/cfg['experiment_id']/'tables'
if (p/'metrics.json').exists():print((p/'metrics.json').read_text())
if (p/'per_second.csv').exists():display(__import__('pandas').read_csv(p/'per_second.csv').head())

## Vídeo anotado
Execute `scripts/anotar_video.py VIDEO PREDICTIONS.csv SAIDA.mp4`. Os tempos seguem PTS e ausências continuam visíveis. Não usa valores OOF ou referência fora da execução selecionada.